# Module 2 lab · Vectors

The code from every lesson of [this module](https://aicourses.dev/mathematics/vectors), in order. Read a lesson, then run its cells here: each lesson builds on the one before, so run the cells from the top the first time.

**First time?** Read the [one-time setup](https://aicourses.dev/mathematics/labs): a Google account, your own copy, and what to expect.

Under each cell, *The lesson shows* is what it should print. Your last digits can differ a little.

**Keep your work:** File → Save a copy in Drive. Colab does not keep changes to this copy.

## 2.2 Adding and scaling: the two moves

Adding and scaling are the two moves that everything in linear algebra is built from. Once you have them, matrices in Module 03 are a way of doing both at scale.

[Read the lesson](https://aicourses.dev/mathematics/vectors/adding-and-scaling)

### In NumPy

In NumPy an array is a list of numbers, and `+`, `-` and `*` by a plain number work on every position at once, as lesson 00.3 showed. First a loop that adds one position at a time, then the one-line version:

In [ ]:
import numpy as np

v = np.array([3, 4])
w = np.array([1, 2])
loop = np.array([v[i] + w[i] for i in range(2)])   # one position at a time
print(loop, v + w)
print(2 * v, -w)
print(2 * v - 3 * w)

*The lesson shows:*

```text
[4 6] [4 6]
[6 8] [-1 -2]
[3 2]
```

A real use is an average. The course’s twelve penguins each have a vector, flipper length and body mass. Add all twelve and divide by 12, that is, scale by one twelfth, and you get the average bird:

In [ ]:
flipper = np.array([181, 183, 185, 192, 193, 193,
                    197, 198, 205, 208, 216, 222])
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350])
birds = [np.array([f, m]) for f, m in zip(flipper, mass)]
total = sum(birds)                 # add all twelve, tip to tail
print(total, total / 12)

*The lesson shows:*

```text
[ 2373 46350] [ 197.75 3862.5 ]
```

## 2.3 Arrays and shapes in NumPy

Before more maths, you need to hold a vector in NumPy and read its shape without being surprised. The shape rules here are the ones that will trip you up in every later lesson that uses arrays.

[Read the lesson](https://aicourses.dev/mathematics/vectors/arrays-and-shapes)

### A list and a column are different

In [ ]:
import numpy as np

v = np.array([3, 4, 6])
col = v.reshape(3, 1)
print(v.shape, col.shape)
print(v.ndim, col.ndim)
print(v + v)
print((v + col).shape)
print(v + col)

*The lesson shows:*

```text
(3,) (3, 1)
1 2
[ 6  8 12]
(3, 3)
[[ 6  7  9]
 [ 7  8 10]
 [ 9 10 12]]
```

### One operation on a whole table

Now the digits. Each picture is a list of 64 numbers, and the 1,297 training pictures stacked one per row make a table of shape `(1297, 64)`. Here is the split every module uses: it shuffles once with a fixed seed, scales the brightness to the range 0 to 1, and keeps 500 pictures aside for testing.

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
print(Xtr.shape, ytr.shape, Xte.shape)
print(Xtr[0].shape, Xtr[:3].shape)
print(np.round(Xte[0].reshape(8, 8) * 16).astype(int))

*The lesson shows:*

```text
(1297, 64) (1297,) (500, 64)
(64,) (3, 64)
[[ 0  0  0  4 11  9  5  0]
 [ 0  0  5 16 16 16  5  0]
 [ 0  0 11 16 16  9  0  0]
 [ 0  4 16 16 16  4  0  0]
 [ 0  1 14 16  9  0  0  0]
 [ 0  4 15 16  6  0  0  0]
 [ 0  0  9 16  8  0  0  0]
 [ 0  0  0  7  5  0  0  0]]
```

To do one thing to every row, you do not need a loop. Take the average picture, the mean of each column, and take it off every row:

In [ ]:
mean = Xtr.mean(axis=0)               # one number per square
print(mean.shape)
centred = Xtr - mean                  # the mean is taken off every row
print(centred.shape)
one = np.array([row - mean for row in Xtr])   # a loop, one row at a time
print(np.allclose(centred, one))
print(np.abs(centred.mean(axis=0)).max() < 1e-12)

*The lesson shows:*

```text
(64,)
(1297, 64)
True
True
```

### The first broadcast

For `(1297, 64)` against `(64,)`: the right places are 64 and 64, equal. The left place has 1297 against nothing, which counts as 1, so the list is stretched down. Now the failing case from NumPy’s guide. Take each picture’s total ink with `sum(axis=1)`, which sums across the columns and leaves 1,297 numbers, and try to subtract it:

In [ ]:
sizes = Xtr.sum(axis=1)              # total ink of each picture
print(sizes.shape)
try:
    Xtr - sizes
except ValueError as e:
    print(e)
share = Xtr / sizes.reshape(-1, 1)   # each square as a share of its ink
print(share.shape, share.sum(axis=1)[:3])

*The lesson shows:*

```text
(1297,)
operands could not be broadcast together with shapes (1297,64) (1297,) 
(1297, 64) [1. 1. 1.]
```

### Your turn: will it fit?

In [ ]:
A = np.zeros((4, 3))
for shape in [(3,), (4,), (4, 1)]:
    try:
        print(shape, (A + np.zeros(shape)).shape)
    except ValueError:
        print(shape, "error")

*The lesson shows:*

```text
(3,) (4, 3)
(4,) error
(4, 1) (4, 3)
```

## 2.4 How long is a vector? Norms

How long is a vector? This lesson answers in any number of dimensions, and the answer is the ruler used for distance, for unit vectors and for weight decay in Module 08.

[Read the lesson](https://aicourses.dev/mathematics/vectors/length-and-norms)

### Three rulers

- **L2**, the straight line: $\sqrt{9 + 16} = 5$.
- **L1**, the sum of the sizes, ignoring signs: $3 + 4 = 7$. It is the walk along the grid, east then north, as a taxi must drive.
- **Max**, the biggest single size: 4.

In [ ]:
import numpy as np

v = np.array([3.0, 4.0])
total = 0.0
for x in v:                     # square each entry, add them up
    total += x * x
print(total ** 0.5)
print(np.linalg.norm(v), np.linalg.norm(v, 1), np.abs(v).max())
print(np.linalg.norm(np.ones(64)))

*The lesson shows:*

```text
5.0
5.0 7.0 4.0
8.0
```

### Unit vectors

A second, closely related ruler is the **root mean square**, or RMS: square the numbers, take their average rather than their sum, and then the root. For $(3, 4)$ the average of the squares is $(9 + 16) / 2 = 12.5$ and the RMS is $\sqrt{12.5} = 3.536$. It is the L2 length divided by $\sqrt{n}$, here $5 / \sqrt{2}$.

In [ ]:
u = v / np.linalg.norm(v)
print(u, np.linalg.norm(u))
rms = np.sqrt((v ** 2).mean())
print(rms, v / rms)

*The lesson shows:*

```text
[0.6 0.8] 1.0
3.5355339059327378 [0.84852814 1.13137085]
```

### Your turn: a length by hand

In [ ]:
a = np.array([2.0, -3.0, 6.0])
print(np.linalg.norm(a), np.linalg.norm(a, 1), np.abs(a).max())

*The lesson shows:*

```text
7.0 11.0 6.0
```

Now the digits, with the fixed split every module uses. Each row of `Xtr` is a picture, and `axis=1` gives one length per row:

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
norms = np.linalg.norm(Xtr, axis=1)
print(norms.shape, norms.min().round(3), norms.max().round(3))
print(norms.mean().round(3), np.linalg.norm(Xte[0]).round(3))

*The lesson shows:*

```text
(1297,) 2.927 4.745
3.863 4.01
```

## 2.5 How far apart? Distance

Two things are alike when the vector between them is short. Distance is the length of a difference, and it is the first way you will tell one digit from another.

[Read the lesson](https://aicourses.dev/mathematics/vectors/distance)

### Distance is the length of a difference

In [ ]:
import numpy as np

a = np.array([3.0, 4.0])
b = np.array([1.0, 2.0])
diff = a - b
print(diff, np.linalg.norm(diff), (diff ** 2).sum())

*The lesson shows:*

```text
[2. 2.] 2.8284271247461903 8.0
```

### Squared distance and the error you know

You have met this already. The mean squared error of lesson 00.5 is the average of the squared misses. The misses are the difference between a vector of predictions and a vector of true values, so the sum of squared misses is a squared distance between those two vectors, and the error is that divided by the number of penguins. With the flat guess of 4,000 g for all twelve penguins, the error was 564,271:

In [ ]:
mass = np.array([3625, 3550, 3150, 2700, 2925, 3650,
                 3750, 4400, 4300, 4200, 4750, 5350])
guess = np.full(12, 4000.0)         # the flat guess, one number per bird
print(round(np.linalg.norm(mass - guess) ** 2 / 12, 1))
print(round(((mass - guess) ** 2).mean(), 1))

*The lesson shows:*

```text
564270.8
564270.8
```

### Your turn: a distance by hand

In [ ]:
a = np.array([1.0, 5.0, 2.0])
b = np.array([4.0, 1.0, 14.0])
print(np.linalg.norm(b - a), np.linalg.norm(a - b))

*The lesson shows:*

```text
13.0 13.0
```

### Which digit is nearest?

Now the digits, with the split every module uses. Take the first test picture, a 1. A loop measures its distance to each training digit, one at a time; the NumPy line does all 1,297 subtractions at once, because `Xtr - x` stretches `x` down every row (lesson 02.3):

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
x = Xte[0]
loop = np.array([np.linalg.norm(t - x) for t in Xtr])
d = np.linalg.norm(Xtr - x, axis=1)
print(np.allclose(d, loop), yte[0])

*The lesson shows:*

```text
True 1
```

Both ways agree. The array `d` holds 1,297 distances, one per training digit. Which is smallest, and what are its neighbours?

In [ ]:
nearest = d.argmin()
print(nearest, ytr[nearest], d[nearest].round(3))
print(np.sort(d)[:5].round(3), ytr[np.argsort(d)[:5]])
print(d.max().round(3), ytr[d.argmax()])
pred = np.array([ytr[np.linalg.norm(Xtr - q, axis=1).argmin()]
                 for q in Xte[:10]])
print(pred, yte[:10])

*The lesson shows:*

```text
1122 1 0.923
[0.923 1.146 1.304 1.31  1.352] [1 1 1 1 1]
4.172 2
[1 7 6 5 2 2 9 1 8 5] [1 7 6 5 2 2 9 1 8 5]
```

## 2.6 The dot product

The dot product is the most important operation in the course: it scores how much two vectors agree. It comes back as a neuron’s weighted sum and as the attention score in Module 12.

[Read the lesson](https://aicourses.dev/mathematics/vectors/the-dot-product)

### Multiply matching entries and add

In [ ]:
import numpy as np

a = np.array([3, 4])
b = np.array([1, 2])
total = 0
for i in range(2):              # multiply matching entries, add them up
    total += a[i] * b[i]
print(total, a @ b, (a * b).sum())
print(a * b)

*The lesson shows:*

```text
11 11 11
[3 8]
```

### The sign: same way, across or against

Multiplying out the brackets as in school algebra gives $\mathbf{a} \cdot \mathbf{a} - \mathbf{a} \cdot \mathbf{b} - \mathbf{b} \cdot \mathbf{a} + \mathbf{b} \cdot \mathbf{b}$, and $\mathbf{a} \cdot \mathbf{b}$ equals $\mathbf{b} \cdot \mathbf{a}$, so the two middle terms make $-2\,\mathbf{a} \cdot \mathbf{b}$. Worked for $(3, 4)$ and $(1, 2)$: $25 + 5 - 2 \times 11 = 8$, the squared distance you found in lesson 02.5. The dot product is the only part of this that depends on how the two vectors relate; the lengths are each one’s own. Lesson 02.9 will use this to explain a surprise.

In [ ]:
print(a @ a, np.linalg.norm(a) ** 2)
print(a @ a + b @ b - 2 * (a @ b), ((a - b) ** 2).sum())
for c in ([4, -3], [1, 2], [-1, -2]):
    print(c, a @ np.array(c))

*The lesson shows:*

```text
25 25.0
8 8
[4, -3] 0
[1, 2] 11
[-1, -2] -11
```

### Your turn: a dot product by hand

In [ ]:
print(np.array([1, 2, 3]) @ np.array([-2, 0, 4]))

*The lesson shows:*

```text
10
```

### Digits, with a dot product

With the split every module uses, take the first test digit, the 1 from lesson 02.1, its nearest training digit, and a training 0:

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
x = Xte[0]
near = Xtr[np.linalg.norm(Xtr - x, axis=1).argmin()]
zero = Xtr[ytr == 0][0]
print((x @ x).round(3), (x @ near).round(3), (x @ zero).round(3))
print((near @ near).round(3), np.linalg.norm(x - near).round(3))

*The lesson shows:*

```text
16.082 15.918 10.379
16.605 0.923
```

## 2.7 Angle and cosine similarity

Dividing the dot product by both lengths leaves only direction, which is what many AI systems compare. The lesson also shows when that is the wrong thing to throw away.

[Read the lesson](https://aicourses.dev/mathematics/vectors/cosine-similarity)

### Divide out the lengths

In [ ]:
import numpy as np

a = np.array([3.0, 4.0])
b = np.array([1.0, 2.0])
cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print(cos.round(5), np.degrees(np.arccos(cos)).round(2))
print(a @ b, (2 * a) @ b)
ua, ub = a / np.linalg.norm(a), b / np.linalg.norm(b)
print((2 * a) @ b / (np.linalg.norm(2 * a) * np.linalg.norm(b)), ua @ ub)

*The lesson shows:*

```text
0.98387 10.3
11.0 22.0
0.9838699100999074 0.9838699100999074
```

### Your turn: a cosine by hand

In [ ]:
a = np.array([1.0, 2.0, 2.0])
b = np.array([2.0, 3.0, 6.0])
cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print(cos.round(4), np.degrees(np.arccos(cos)).round(2))

*The lesson shows:*

```text
0.9524 17.75
```

### What cosine hides

Here the digits show the effect. Take the first test digit and its nearest training digit, and then dim the test digit to half its brightness:

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
x = Xte[0]
near = Xtr[np.linalg.norm(Xtr - x, axis=1).argmin()]
cos = lambda p, q: p @ q / (np.linalg.norm(p) * np.linalg.norm(q))
for f in (1.0, 0.5):
    print(f, cos(f * x, near).round(4), (f * x @ near).round(3),
          np.linalg.norm(f * x - near).round(3))

*The lesson shows:*

```text
1.0 0.9741 15.918 0.923
0.5 0.9741 7.959 2.17
```

## 2.8 Projection: the shadow of one vector on another

Projection asks how much of one vector lies along another. It builds directly on the dot product and returns as least squares in Module 10.

[Read the lesson](https://aicourses.dev/mathematics/vectors/projection)

### What is left over

The leftover is perpendicular to $\mathbf{b}$. You can check in numbers: its dot product with $\mathbf{b}$ is $-0.32 \times 3 + 0.24 \times 4 = -0.96 + 0.96 = 0$. A dot product of zero is the sign of right angles (lesson 02.6). And Pythagoras holds: the squared length of $\mathbf{a}$ is $5$, the shadow’s is $4.84$ and the leftover’s is $0.1024 + 0.0576 = 0.16$, which add to 5. This can be proved for any vectors; here you check it.

In [ ]:
import numpy as np

a = np.array([1.0, 2.0])
b = np.array([3.0, 4.0])
k = (a @ b) / (b @ b)
shadow = k * b
left = a - shadow
print(k, shadow, left)
print(round(float(left @ b), 10) + 0.0)
print((shadow @ shadow).round(2), (left @ left).round(2), a @ a)
u = b / np.linalg.norm(b)
print(a @ u, (a @ u) * u)

*The lesson shows:*

```text
0.44 [1.32 1.76] [-0.32  0.24]
0.0
4.84 0.16 5.0
2.2 [1.32 1.76]
```

### Your turn: a projection by hand

In [ ]:
a = np.array([7.0, 1.0])
b = np.array([2.0, 1.0])
shadow = (a @ b) / (b @ b) * b
print(shadow, a - shadow, (a - shadow) @ b)

*The lesson shows:*

```text
[6. 3.] [ 1. -2.] 0.0
```

### A digit onto another digit

The same recipe works with 64 numbers. With the split every module uses, project the first test digit onto its nearest training digit:

In [ ]:
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out
x = Xte[0]
b = Xtr[np.linalg.norm(Xtr - x, axis=1).argmin()]
k = (x @ b) / (b @ b)
shadow, left = k * b, x - k * b
print(k.round(4), np.linalg.norm(shadow).round(3),
      np.linalg.norm(left).round(3))
print(round(float(left @ b), 10) + 0.0)

*The lesson shows:*

```text
0.9586 3.906 0.907
0.0
```

## 2.9 Project: recognise digits by their nearest neighbour

You label test digits by their closest training digit, score two ways of measuring closeness and then break the classifier on purpose. It is the module’s first working model and the baseline for Module 03’s.

[Read the lesson](https://aicourses.dev/mathematics/vectors/project-nearest-neighbours)

**This is the project.** These cells are starting points from the lesson; the steps between them are yours to write. A cell that uses a name from one of your steps stops with a NameError until you have written that step.

### Step 1: one test digit

The same search for one real test digit, in a loop: measure the distance to every training digit and keep the smallest. The block begins with the fixed split of every module.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits

rng = np.random.default_rng(0)
X, y = load_digits(return_X_y=True)
X = X / 16                      # pixel values 0 to 1
order = rng.permutation(len(X))  # one seeded shuffle
X, y = X[order], y[order]
Xtr, ytr = X[:1297], y[:1297]    # 1,297 to learn from
Xte, yte = X[1297:], y[1297:]    # 500 held out

q = Xte[0]                       # one test digit
best, best_d = -1, 1e9
for i in range(len(Xtr)):        # compare with every training digit
    d = np.sqrt(((Xtr[i] - q) ** 2).sum())
    if d < best_d:
        best, best_d = i, d
print(best, ytr[best], round(best_d, 3), "true label", yte[0])

*The lesson shows:*

```text
1122 1 0.923 true label 1
```

### Step 2: all 500 test digits

Now make “nearest” into functions that take a test digit and return the position of its nearest training digit. The distance version does all 1,297 subtractions at once (lesson 02.3). For cosine, `(Xtr * q).sum(axis=1)` multiplies every training row by `q` entry by entry and adds along each row: all 1,297 dot products at once.

In [ ]:
def nearest_by_distance(q):
    return np.linalg.norm(Xtr - q, axis=1).argmin()

def nearest_by_cosine(q):
    dots = (Xtr * q).sum(axis=1)          # a dot product with every row
    lengths = np.linalg.norm(Xtr, axis=1) * np.linalg.norm(q)
    return (dots / lengths).argmax()      # the largest cosine, not smallest

def accuracy(nearest, queries):
    pred = np.array([ytr[nearest(q)] for q in queries])
    return (pred == yte).mean()

print(accuracy(nearest_by_distance, Xte))
print(accuracy(nearest_by_cosine, Xte))

*The lesson shows:*

```text
0.992
0.988
```

### Step 3: dim some of the test digits

Real pictures differ in brightness. Dim the first 250 test digits, leaving the other 250 alone, by multiplying them by 0.5 (`Xte.copy()` makes a separate table, so `Xte` itself stays clean; `*= f` multiplies in place; and `!=` in a later block marks the wrong labels). Then score again, and try 0.25 and 0.1 as well.

In [ ]:
for f in (0.5, 0.25, 0.1):
    dim = Xte.copy()
    dim[:250] *= f               # the first 250 test digits, dimmed
    print(f, accuracy(nearest_by_distance, dim),
          accuracy(nearest_by_cosine, dim))

*The lesson shows:*

```text
0.5 0.936 0.988
0.25 0.714 0.988
0.1 0.548 0.988
```

### Why the dimmed digits fail

Distance has a different story. Expand it with the identity from lesson 02.6: the squared distance from a test digit $\mathbf{q}$ to a training digit $\mathbf{t}$ is $\lVert \mathbf{q} \rVert^2 + \lVert \mathbf{t} \rVert^2 - 2\,\mathbf{q} \cdot \mathbf{t}$. The first term is the same for every training digit, so it does not change which is nearest. The training digit that wins is the one with the smallest $\lVert \mathbf{t} \rVert^2 - 2\,\mathbf{q} \cdot \mathbf{t}$: a reward for matching, the dot product, set against a penalty for being long. Halve $\mathbf{q}$ and the reward halves while the penalty stays. Short training digits now beat long ones. Check it:

In [ ]:
lens = np.linalg.norm(Xtr, axis=1)
a = np.array([nearest_by_distance(q) for q in Xte[:250]])
b = np.array([nearest_by_distance(0.5 * q) for q in Xte[:250]])
print(lens.mean().round(3), lens[a].mean().round(3), lens[b].mean().round(3))
print((ytr[a] != yte[:250]).sum(), (ytr[b] != yte[:250]).sum())
q = 0.5 * Xte[0]
for k in (nearest_by_distance(Xte[0]), nearest_by_distance(q)):
    print(ytr[k], lens[k].round(3), np.linalg.norm(q - Xtr[k]).round(3))

*The lesson shows:*

```text
3.863 3.869 3.363
0 28
1 4.075 2.17
8 3.452 1.845
```